<a href="https://colab.research.google.com/github/Gallicchio-Lab/AToM-OpenMM/blob/master/example-notebooks/ABFE/atom_openmm_abfe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Example of Protein-ligand Alchemical Absolute Binding Free Energy Calculation with AToM-OpenMM

### Acknowledgements

- OpenMM: https://openmm.org
- openmmforcefields: https://github.com/openmm/openmmforcefields
- Eric Chen's ATM implementation: https://github.com/EricChen521/atm
- Making-it-rain notebooks: https://github.com/pablo-arantes/making-it-rain
- FKBP ABFE system from the AToM-OpenMM examples
- Thanks to Stefan Doerr @Acellera for helping modernize the atom-openmm modules.

### Procedure

- In this example, the FKBP receptor and the `but.sdf` ligand from the AToM-OpenMM examples are combined into a complex.
- The ligand is displaced into solvent by an automatically chosen displacement vector.
- A one-atom noninteracting ghost residue is placed at the original bound ligand attachment point.
- The system is solvated, minimized, thermalized, equilibrated, and annealed at the alchemical intermediate.
- Then an alchemical Hamiltonian replica exchange simulation is carried out.
- Finally, the perturbation energy data collected during the simulation is analyzed to estimate the absolute binding free energy.

Read the [introduction to AToM-OpenMM](https://www.compmolbiophysbc.org/atom-openmm) to get a better sense of the steps involved.

### To run the notebook

- If the `Open in Colab` badge does not work yet, open [https://colab.research.google.com](https://colab.research.google.com), choose `GitHub` or `Upload`, and then either paste the notebook URL or upload this `.ipynb` file directly.

- On Google Colab, connect to a GPU runtime.
- Execute the first cell to install `condacolab`. The runtime will restart. When done, continue with the following cells.
- The final cell can zip and download the simulation results.

### **Install Conda Colab (if on Google Colab)**

It will restart the kernel; the warning is harmless.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install_from_url("https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh")

### **Install dependencies**

In [ ]:
import subprocess
import sys

# Fixes sys.path that can trigger a torchvision import error on Colab after condacolab restarts.
original_syspath = sys.path.copy()
new_syspath = [
    '/content', '/env/python', '/usr/local/lib/python312.zip',
    '/usr/local/lib/python3.12', '/usr/local/lib/python3.12/lib-dynload',
    '/usr/local/lib/python3.12/site-packages'
]
sys.path = new_syspath

subprocess.run(
    "mamba install openmm openmmforcefields openff-toolkit ambertools rdkit "
    "pandas pyyaml matplotlib setproctitle -y",
    shell=True,
    check=True,
)

# Clone upstream by default, or point to a fork/branch when testing notebook changes.
repo_owner = 'Gallicchio-Lab' # 'eliantiudic' for testing
repo_name = 'AToM-OpenMM'
repo_ref = '' # 'abfe-variable-displacement' for testing  # e.g. 'some-feature-branch' when testing unreleased notebook changes
repo_url = f'https://github.com/{repo_owner}/{repo_name}.git'
clone_cmd = f'git clone {repo_url}' if not repo_ref else f'git clone -b {repo_ref} {repo_url}'
subprocess.run(f"cd /content && {clone_cmd}", shell=True, check=True)
subprocess.run("cd /content/AToM-OpenMM && pip install .", shell=True, check=True)

!pip install py3Dmol

### **Test the OpenMM installation**

In [ ]:
import openmm.testInstallation
openmm.testInstallation.main()

### **Select the FKBP input files**

The notebook uses local copies of the FKBP receptor and `but.sdf` ligand from `example-notebooks/ABFE/data`.

In [ ]:
import os
from pathlib import Path

def find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/AToM-OpenMM')]
    for path in candidates:
        data_dir = path / 'example-notebooks' / 'ABFE' / 'data'
        if (path / 'atom_openmm').exists() and (data_dir / 'fkbp.pdb').exists() and (data_dir / 'but.sdf').exists():
            return path
    raise FileNotFoundError('Could not find the AToM-OpenMM repository root with example-notebooks/ABFE/data.')

#collect names of receptor pdb file and ligand sdf files
Protein_PDB_file_name = 'fkbp.pdb'
Ligand1_SDF_file_name = 'but.sdf'

repo_root = find_repo_root()

#where the receptor and ligand file name are stored
dataDir = repo_root / 'example-notebooks' / 'ABFE' / 'data'

#full file names
receptor_file = dataDir / Protein_PDB_file_name
ligand_file = dataDir / Ligand1_SDF_file_name

#basename of the job (the job name)
basename = Path(Protein_PDB_file_name).stem + '-' + Path(Ligand1_SDF_file_name).stem

#create the working directory and change to it
work_dir = (Path('/content') if Path('/content').exists() else Path.cwd()) / basename
work_dir.mkdir(parents=True, exist_ok=True)

#the ATM system xml file and the output pdb file
pdboutfile = work_dir / f'{basename}.pdb'
outxml = work_dir / f'{basename}_sys.xml'
ffcache = work_dir / 'ff.json'

print('Repository:', repo_root)
print('Data directory:', dataDir)
print('Receptor:', receptor_file)
print('Ligand:', ligand_file)
print('Work directory:', work_dir)

### **View the ligand and choose the attachment atom**

The attachment atom defines the bound ghost-particle position. The default value `0` chooses the first non-hydrogen atom automatically. To choose a specific atom, set `ligand_attach_atom_index` to the 1-based atom serial shown below.

In [ ]:
import py3Dmol

with open(ligand_file, 'r') as f:
    ligand_data = f.read()

view = py3Dmol.view(width=800, height=500)
view.addModel(ligand_data, 'sdf')
view.setStyle({}, {'stick': {'radius': 0.12}})
view.addPropertyLabels('serial', {}, {'fontColor': 'red', 'alignment': 'center'})
view.zoomTo()
view.show()

In [ ]:
ligand_attach_atom_index = 0 #@param {type:"integer"}

attach_index = None if ligand_attach_atom_index in (None, 0) else int(ligand_attach_atom_index) - 1

view = py3Dmol.view(width=800, height=500)
view.addModel(ligand_data, 'sdf')
view.setStyle({}, {'stick': {'radius': 0.12}})
if attach_index is not None:
    view.addStyle({'serial': int(ligand_attach_atom_index)}, {'sphere': {'radius': 0.4, 'color': 'orange'}})
view.zoomTo()
view.show()

### **Prepare the ATM system in a box of water**

This follows the FKBP ABFE workflow: build the bound receptor-ligand system, translate the ligand into solvent as `L2`, and append a one-atom noninteracting `L1` ghost at the bound attachment point.

In [ ]:
from openmm import XmlSerializer
from openmm.app import PDBFile
from atom_openmm.make_atm_system_from_rcpt_lig import make_system
from atom_openmm.utils.AtomUtils import calc_displ_vec, patch_system_with_ghost

def system_has_ghost_pair(pdb_file):
    if not Path(pdb_file).exists():
        return False
    pdb = PDBFile(str(pdb_file))
    residue_names = [residue.name for residue in pdb.topology.residues()]
    return residue_names.count('L1') == 1 and residue_names.count('L2') == 1

displacement = list(calc_displ_vec(str(receptor_file), str(ligand_file)))
print('Displacement vector:', displacement, 'Angstroms')

os.chdir(work_dir)

if not outxml.exists() or not pdboutfile.exists() or not system_has_ghost_pair(pdboutfile):
    make_system(
        receptorfile=str(receptor_file),
        lig1file=str(ligand_file),
        displacement=displacement,
        xmloutfile=str(outxml),
        pdboutfile=str(pdboutfile),
        ffcachefile=str(ffcache),
        hmass=1.5,
        ionicstrength=0.15,
        ligandforcefield='gaff',
    )
    patch_system_with_ghost(
        str(pdboutfile),
        str(outxml),
        displacement,
        ghost_mass=12.011,
        attach_index=attach_index,
    )
else:
    print('Using existing prepared ABFE system files.')

### **View the Protein-ligand ABFE system in water**

`L1` is the bound ghost particle and `L2` is the physical ligand displaced into solvent.

In [ ]:
with open(pdboutfile, 'r') as f:
    pdb_data = f.read()

view = py3Dmol.view(width=900, height=650)
view.addModel(pdb_data, 'pdb')
view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'spectrum'}})
view.setStyle({'resn': 'L2'}, {'stick': {'radius': 0.22, 'color': 'orange'}})
view.setStyle({'resn': 'L1'}, {'sphere': {'radius': 0.55, 'color': 'red'}})
view.setStyle({'resn': ['SOL', 'WAT', 'HOH', 'TIP3']}, {})
view.zoomTo({'resn': ['L1', 'L2']})
view.show()

### **Setup Absolute Binding Free Energy Calculation**

This performs energy minimization, thermalization, equilibration, and annealing to the alchemical intermediate at lambda = 1/2. The short sampling settings below are intended for a notebook demonstration.

In [ ]:
import yaml
from openmm import Vec3
from openmm.app import PDBFile
from openmm.unit import angstrom, nanometer
from atom_openmm.rbfe_structprep import rbfe_structprep
from atom_openmm.rbfe_production import rbfe_production
from atom_openmm.utils.AtomUtils import (
    get_attach_atom_from_residue,
    get_indexes_from_query,
    get_indexes_from_residue,
    get_residue_by_name,
    get_selected_principal_groups,
)

options = {
    'JOB_TRANSPORT': 'LOCAL_OPENMM',
    'RE_SETUP': 'YES',
    'TEMPERATURES': [300.0],
    'LAMBDAS':      [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.00],
    'DIRECTION':    [   1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1],
    'INTERMEDIATE': [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    1,    1,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
    'LAMBDA1':      [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.50, 0.40, 0.30, 0.20, 0.10, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
    'LAMBDA2':      [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.40, 0.30, 0.20, 0.10, 0.00],
    'ALPHA':        [0.10] * 22,
    'U0':           [110.0] * 22,
    'W0COEFF':      [0] * 22,
    'WALL_TIME': 240,
    'MAX_SAMPLES': 10,
    'CYCLE_TIME': 10,
    'CHECKPOINT_TIME': 1200,
    'SUBJOBS_BUFFER_SIZE': 1.0,
    'PRODUCTION_STEPS': 5000,
    'PRNT_FREQUENCY': 5000,
    'TRJ_FREQUENCY': 5000,
    'CM_KF': 25.00,
    'CM_TOL': 5,
    'POSRE_FORCE_CONSTANT': 0.0,
    'POSRE_TOLERANCE': 3.5,
    'UMAX': 200.00,
    'ACORE': 0.062500,
    'UBCORE': 100.0,
    'FRICTION_COEFF': 0.500000,
    'HMASS': 1.5,
    'TIME_STEP': 0.004,
    'VERBOSE': False,
    'LIGAND_FORCE_FIELD': 'gaff',
    'GHOST_MASS': 12.011,
    'BASENAME': basename,
    'WORKDIR': str(work_dir),
}
if attach_index is not None:
    options['LIGAND_ATTACH_INDEX'] = attach_index

pdb = PDBFile(str(pdboutfile))
topology = pdb.topology
positions = pdb.positions

ghost_residue = get_residue_by_name(topology, 'L1')
ligand_residue = get_residue_by_name(topology, 'L2')
assert ghost_residue is not None
assert ligand_residue is not None

ligand1_atoms = get_indexes_from_residue(ghost_residue)
ligand2_atoms = get_indexes_from_residue(ligand_residue)
options['LIGAND1_ATOMS'] = ligand1_atoms
options['LIGAND2_ATOMS'] = ligand2_atoms
options['LIGAND1_VAR_ATOMS'] = ligand1_atoms
options['LIGAND2_VAR_ATOMS'] = ligand2_atoms

ghost_attach_atom = list(ghost_residue.atoms())[0]
ligand_attach_atom = get_attach_atom_from_residue(ligand_residue, attach_index=attach_index)
options['LIGAND1_ATTACH_ATOM'] = ghost_attach_atom.index
options['LIGAND2_ATTACH_ATOM'] = ligand_attach_atom.index
options['LIGAND1_CM_ATOMS'] = [ghost_attach_atom.index]
options['LIGAND2_CM_ATOMS'] = [ligand_attach_atom.index]

lig1cm_pos = positions[ghost_attach_atom.index]
lig2cm_pos = positions[ligand_attach_atom.index]
displ = (lig2cm_pos - lig1cm_pos).value_in_unit(angstrom)
options['DISPLACEMENT'] = [displ.x, displ.y, displ.z]

rcpt_chain_name = options.get('RCPT_CHAIN_NAME', 'A')
rcpt_frame_query = f'atom.residue.chain.id == "{rcpt_chain_name}" and atom.name == "CA"'
rcpt_frame_indexes = get_indexes_from_query(topology, rcpt_frame_query)
rcpt_frame = get_selected_principal_groups(topology, positions, rcpt_frame_indexes)

options['RCPT_CM_ATOMS'] = rcpt_frame['origin']['indices']
options['RCPT_FRAME_ATOMS_O'] = options['RCPT_CM_ATOMS']
options['RCPT_FRAME_ATOMS_Z'] = rcpt_frame['z_axis']['indices']
options['RCPT_FRAME_ATOMS_Y'] = rcpt_frame['y_axis']['indices']

rcpt_cm_pos = Vec3(
    float(rcpt_frame['origin']['com'][0]),
    float(rcpt_frame['origin']['com'][1]),
    float(rcpt_frame['origin']['com'][2]),
) * nanometer
offset = (lig1cm_pos - rcpt_cm_pos).value_in_unit(angstrom)
options['LIGOFFSET'] = [offset.x, offset.y, offset.z]

options['POS_RESTRAINED_ATOMS'] = None
options['EXCLUSION_POT_MOL1_INDEXES'] = get_indexes_from_query(
    topology,
    f'(atom.residue.chain.id == "{rcpt_chain_name}") and (atom.element.atomic_number != 1)',
)
options['EXCLUSION_POT_MOL2_INDEXES'] = get_indexes_from_residue(
    ligand_residue,
    query='(atom.element.atomic_number != 1)',
)

with open(work_dir / f'{basename}.yaml', 'w') as f:
    yaml.dump(options, f, default_flow_style=None, width=1000000, sort_keys=False)

print('ATM ABFE SETTINGS:')
print(options)

if not (work_dir / f'{basename}_0.xml').exists():
    rbfe_structprep(config_file=None, options=options)
else:
    print('Using existing structural preparation files.')

### **Run Absolute Binding Free Energy Calculation**

This demonstration run collects only a small number of samples. Increase `MAX_SAMPLES` and `WALL_TIME` above for a production-quality estimate.

In [ ]:
rbfe_production(config_file=None, options=options)

### **Analysis: Binding free energy estimate**

Runs UWHAM to estimate the absolute binding free energy. By default, one third of the early data is discarded for equilibration. A much longer simulation is needed for a reliable estimate; this is only an example.

In [ ]:
from atom_openmm.uwham import calculate_uwham_from_rundir, create_quality_assessment_plot

dgb, dgb_std, uwham_data = calculate_uwham_from_rundir(
    options['WORKDIR'],
    options['BASENAME'],
)
print(f'{basename}: DGb = {dgb:.3f} +/- {dgb_std:.3f} kcal/mol')

df1 = uwham_data['df_leg1']
n1 = len(uwham_data['uwham_out_leg1']['W'][:, 0])
df1['W'] = uwham_data['uwham_out_leg1']['W'][:, 0] / float(n1)

df2 = uwham_data['df_leg2']
n2 = len(uwham_data['uwham_out_leg2']['W'][:, 0])
df2['W'] = uwham_data['uwham_out_leg2']['W'][:, 0] / float(n2)

fig = create_quality_assessment_plot(df1, df2)

### **Download the results as a zip file**

In [ ]:
zipfile = work_dir.with_suffix('.zip')
!cd {work_dir.parent} && zip -r {zipfile.name} {work_dir.name}

try:
    from google.colab import files
    files.download(str(zipfile))
except ImportError:
    print('Results zip:', zipfile)